# 🎛️ Steer your model — live feature intervention

**Q1-notebook preview of the Q2 Sandbox.**

Pick a feature ID, pick an α, watch the model's generation bend. This notebook loads your trained Sparse Autoencoder, installs a forward hook on the target layer, and rewrites the residual stream at inference time:

```
h' = h + (SAE.decode(scale(z, feature_id, α)) - SAE.decode(z))
```

We run the same prompt under 4 values of α (ablation → baseline → amplification) and render the outputs side-by-side. The resulting `interventions.json` is the raw material for the Trace Theater counterfactual panel.

**Runs on**: Colab free T4 for ≤4B base models, bf16 + SDPA (no flash-attn).

In [ ]:
# Install deps (Colab)
!pip install -q --upgrade transformers accelerate safetensors huggingface_hub rich

In [ ]:
# === USER CONFIG — edit these ===
HF_SAE_REPO    = "caiovicentino1/openinterp-sae-demo"   # where your SAE lives
HF_BASE_MODEL  = "google/gemma-2-2b-it"                  # the base model the SAE was trained on
LAYER          = 12                                      # layer the SAE was trained on (residual stream)
D_MODEL        = 2304                                    # hidden dim of base model
D_SAE          = 16384                                   # SAE dictionary size
K              = 32                                      # TopK activation

# Pick a feature to steer. Default = first "interesting" feature from catalog (loaded below).
# Override this integer after you've seen the top-5 catalog print.
FEATURE_ID     = 0

PROMPT         = "The best way to learn a new language is"
MAX_NEW_TOKENS = 40

# The four α we contrast in the side-by-side view:
#   -3.0 = strong suppression (negative reconstruction)
#    0.0 = feature ablated (zeroed out)
#    1.0 = untouched baseline
#    3.0 = amplification
ALPHAS         = [-3.0, 0.0, 1.0, 3.0]

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32
print(f"device={DEVICE} dtype={DTYPE}")

## Auth

We need an HF token that (a) can read the base model and (b) can write to `HF_SAE_REPO` for the final `interventions.json` upload.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    # Local / non-Colab: HF_TOKEN must already be in env
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN env var or Colab secret"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF auth ok")

In [ ]:
# Load SAE + base model
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---- Minimal TopK SAE (must match the one used for training) ----
class TopKSAE(nn.Module):
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.d_model, self.d_sae, self.k = d_model, d_sae, k
        self.W_enc = nn.Parameter(torch.empty(d_model, d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_sae, d_model))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        # x: (..., d_model) -> z: (..., d_sae) with TopK sparsity applied
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        # TopK: keep top-k per token, zero the rest
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, torch.relu(topk_vals))
        return z

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

# ---- Download SAE weights ----
sae_path = hf_hub_download(repo_id=HF_SAE_REPO, filename="sae.safetensors")
sd = load_file(sae_path)
sae = TopKSAE(D_MODEL, D_SAE, K).to(DEVICE, dtype=DTYPE)
# Tolerant load: accept either raw param names or wrapped
clean = {k.replace("sae.", ""): v for k, v in sd.items()}
missing, unexpected = sae.load_state_dict(clean, strict=False)
print(f"SAE loaded: missing={len(missing)} unexpected={len(unexpected)}")
sae.eval()

# ---- Base model ----
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=DTYPE,
    attn_implementation="sdpa",   # NO flash-attn
    device_map=DEVICE,
)
model.eval()
print(f"model loaded: {sum(p.numel() for p in model.parameters())/1e9:.2f} B params")

In [ ]:
# Load feature_catalog.json if present — shows user interesting candidates
import json
from huggingface_hub import hf_hub_download

catalog = None
try:
    cat_path = hf_hub_download(repo_id=HF_SAE_REPO, filename="feature_catalog.json")
    with open(cat_path) as f:
        catalog = json.load(f)
    print(f"Catalog loaded: {len(catalog)} features\n")
    print("Top-5 interesting features you might steer:")
    print("-" * 72)
    # Rank by interestingness score if present, else by activation count
    def score(e):
        return e.get("interestingness", e.get("activation_count", 0))
    entries = sorted(catalog if isinstance(catalog, list) else catalog.values(),
                     key=score, reverse=True)[:5]
    for e in entries:
        fid = e.get("feature_id", e.get("id", "?"))
        lbl = e.get("label", e.get("description", "(unlabeled)"))
        print(f"  feature {fid:>5}  |  {lbl}")
    print("-" * 72)
    # If user left FEATURE_ID=0 and catalog has a default, use the top one
    if FEATURE_ID == 0 and entries:
        suggested = entries[0].get("feature_id", entries[0].get("id", 0))
        print(f"\nFEATURE_ID is 0 — using catalog top pick: {suggested}")
        FEATURE_ID = int(suggested)
except Exception as ex:
    print(f"No feature_catalog.json found ({ex.__class__.__name__}) — using FEATURE_ID={FEATURE_ID} as-is")

print(f"\n>>> Will steer feature #{FEATURE_ID} on layer {LAYER} <<<")

## The steering mechanism — key explanation

The trick is to **preserve the SAE's reconstruction error**. A naïve approach — "replace hidden state with `SAE.decode(scale(z))`" — would dump the reconstruction residual into every token, polluting the generation.

Instead we compute the **delta** between modified and unmodified reconstruction, and add *only that delta* to the real hidden state:

$$
h' \;=\; h \;+\; \underbrace{\big(\text{SAE.decode}(z_{\text{mod}}) - \text{SAE.decode}(z)\big)}_{\text{isolated edit}}
$$

Properties:
- At **α=1** the delta is exactly zero → generation is bit-identical to baseline (our α=1 sanity check in cell 13).
- At **α=0** we *ablate* the feature cleanly without touching other directions.
- At **α=3** we amplify along the feature's decoder direction, which is linear in α so we get predictable, monotonic intervention strength.

In [ ]:
# Forward-hook factory: one hook per (feature_id, alpha)
def make_hook(sae, feature_id, alpha):
    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        # Decompose via SAE
        z = sae.encode(hidden)                    # (B, T, d_sae)
        # Scale the target feature only
        z_mod = z.clone()
        z_mod[..., feature_id] = z[..., feature_id] * alpha
        # Reconstruct + add delta (preserves reconstruction error)
        delta = sae.decode(z_mod) - sae.decode(z)
        new_hidden = hidden + delta
        if isinstance(output, tuple):
            return (new_hidden,) + output[1:]
        return new_hidden
    return hook

# Discover the transformer block list: try `model.language_model.layers.N` (multimodal nesting)
# then fall back to `model.model.layers.N` (standard LLaMA/Gemma/Qwen).
def get_target_block(model, layer_idx):
    candidates = [
        ("model.language_model.layers", lambda m: m.language_model.layers),
        ("model.model.layers",          lambda m: m.model.layers),
        ("model.layers",                lambda m: m.layers),  # already unwrapped
    ]
    errors = []
    for name, getter in candidates:
        try:
            blocks = getter(model)
            block  = blocks[layer_idx]
            print(f"found block via {name}[{layer_idx}] -> {type(block).__name__}")
            return block
        except (AttributeError, IndexError) as e:
            errors.append(f"{name}: {e.__class__.__name__}")
    raise RuntimeError(f"could not locate layer {layer_idx}. Tried: {errors}")

target_block = get_target_block(model, LAYER)

In [ ]:
# For each alpha: install hook, generate, remove hook, decode. Store all 4 outputs.
import time

inputs = tok(PROMPT, return_tensors="pt").to(DEVICE)
gen_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,          # greedy = deterministic comparison
    temperature=1.0,
    pad_token_id=tok.eos_token_id,
)

results = {}
for alpha in ALPHAS:
    t0 = time.time()
    handle = target_block.register_forward_hook(make_hook(sae, FEATURE_ID, alpha))
    try:
        with torch.no_grad():
            out_ids = model.generate(**inputs, **gen_kwargs)
    finally:
        handle.remove()  # ALWAYS remove, even on error
    text = tok.decode(out_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    results[alpha] = text
    print(f"α={alpha:+.1f}  ({time.time()-t0:.1f}s)  →  {text[:80]}{'...' if len(text) > 80 else ''}")

print("\nAll 4 generations complete.")

In [ ]:
# Pretty-print 4-way side-by-side
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.columns import Columns
    from rich.text import Text
    console = Console(width=140)

    headers = {
        -3.0: ("α = -3.0", "strong suppression", "red"),
         0.0: ("α =  0.0", "feature ablated",   "yellow"),
         1.0: ("α = +1.0", "baseline",          "green"),
         3.0: ("α = +3.0", "amplified",         "magenta"),
    }
    panels = []
    for a in ALPHAS:
        title, subtitle, color = headers.get(a, (f"α = {a:+.1f}", "", "white"))
        body = Text(results[a].strip() or "(empty)")
        panels.append(Panel(body, title=f"[bold {color}]{title}[/]  [dim]{subtitle}[/]",
                            border_style=color, width=34))
    console.print(f"[bold]PROMPT:[/] {PROMPT}")
    console.print(f"[bold]FEATURE:[/] #{FEATURE_ID}  on layer {LAYER}\n")
    console.print(Columns(panels, equal=True, expand=False))
except ImportError:
    # Plain ASCII fallback
    print(f"PROMPT:  {PROMPT}")
    print(f"FEATURE: #{FEATURE_ID}  on layer {LAYER}\n")
    tags = {-3.0: "(strong suppression)", 0.0: "(ablated)",
             1.0: "(baseline)",            3.0: "(amplified)"}
    for a in ALPHAS:
        print("=" * 72)
        print(f"α = {a:+.1f}   {tags.get(a, '')}")
        print("-" * 72)
        print(results[a].strip() or "(empty)")
    print("=" * 72)

## Sanity check — delta should be zero at α=1, nonzero otherwise

The hook adds `decode(z_mod) - decode(z)`. When α=1, `z_mod == z`, so the delta is the zero vector and the hooked model is mathematically identical to the unhooked model. Any other α should produce a nonzero hidden-state delta that grows **monotonically with |α−1|** (because `decode` is linear in the scaled coordinate).

If this curve is flat or non-monotonic, the SAE is broken, the feature is dead at this layer, or the hook isn't firing.

In [ ]:
# Measure ‖h_α − h_baseline‖ in hidden space at the hooked layer, for one forward pass.
# We capture the post-hook hidden state using a second (passive) hook and compare.

import torch

captured = {}
def capture_hook(tag):
    def hook(module, inputs, output):
        h = output[0] if isinstance(output, tuple) else output
        captured[tag] = h.detach().float().clone()
    return hook

norms = {}
for alpha in ALPHAS:
    # Install steering hook first, then capture hook AFTER it — hooks run in registration order,
    # and the capture sees the modified output because the steering hook returns the new tensor.
    h_steer = target_block.register_forward_hook(make_hook(sae, FEATURE_ID, alpha))
    h_cap   = target_block.register_forward_hook(capture_hook(alpha))
    try:
        with torch.no_grad():
            _ = model(**inputs)
    finally:
        h_cap.remove()
        h_steer.remove()

baseline = captured[1.0]
print(f"{'alpha':>6} | {'‖h_α − h_1‖':>16} | {'|α−1|':>6}")
print("-" * 40)
for a in ALPHAS:
    diff = (captured[a] - baseline).norm().item()
    norms[a] = diff
    print(f"{a:>+6.1f} | {diff:>16.4f} | {abs(a-1):>6.2f}")

# Monotonicity check: sort by |α−1|, expect norms to be sorted too
sorted_by_dist = sorted(ALPHAS, key=lambda a: abs(a-1))
monotonic = all(norms[sorted_by_dist[i]] <= norms[sorted_by_dist[i+1]] + 1e-3
                for i in range(len(sorted_by_dist)-1))
assert norms[1.0] < 1e-3, f"α=1 delta should be ~0, got {norms[1.0]}"
print(f"\n✓ α=1 delta is ~0 ({norms[1.0]:.2e})")
print(f"{'✓' if monotonic else '✗'} Monotonic with |α−1|: {monotonic}")

In [ ]:
# Save interventions.json for the Trace Theater counterfactual panel, then upload.
import json, os, tempfile
from huggingface_hub import HfApi

payload = {
    "base_model":      HF_BASE_MODEL,
    "sae_repo":        HF_SAE_REPO,
    "layer":           LAYER,
    "feature_id":      int(FEATURE_ID),
    "prompt":          PROMPT,
    "max_new_tokens":  MAX_NEW_TOKENS,
    "baseline":        results[1.0],
    "alphas": [
        {"alpha": float(a),
         "generation": results[a],
         "hidden_delta_norm": float(norms[a])}
        for a in ALPHAS
    ],
    "schema_version":  "1.0",
}

out_path = os.path.join(tempfile.gettempdir(), "interventions.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print(f"wrote {out_path} ({os.path.getsize(out_path)} bytes)")

# Upload to the SAE repo so Trace Theater can pick it up
try:
    api = HfApi()
    api.upload_file(
        path_or_fileobj=out_path,
        path_in_repo=f"interventions/feature_{FEATURE_ID:05d}.json",
        repo_id=HF_SAE_REPO,
        repo_type="model",
        commit_message=f"steer: feature {FEATURE_ID} @ layer {LAYER}",
    )
    print(f"✓ uploaded to {HF_SAE_REPO}/interventions/feature_{FEATURE_ID:05d}.json")
except Exception as ex:
    print(f"upload skipped: {ex.__class__.__name__}: {ex}")
    print(f"(local copy still at {out_path})")